# 08 — Run Exp 1 (PE-level): LapPE (sign-aligned) vs. fix-L-HKS @ K=8

Runs `ANALYSIS/perturbation/pe_stability_exp1_pe_drift.py`. The observable is **PE drift**

  δ_PE(G, G') = ||PE(G) - PE(G')||_F / ||PE(G)||_F

directly on the positional encoding tensor — no model forward pass, no prediction std. Both PEs are descriptor-intrinsic at the same truncation (first 8 nontrivial eigenvectors), with K = 8 analytic fixed diffusion times `t_j = exp(linspace(log 0.01, log 100, 8))` for the HKS side.

**Two methods, and only two:**
  - `LapPE-aligned` — truncated Laplacian eigenvectors `EigVecs`, column-wise sign-aligned to the base before differencing.
  - `fix-L-HKS-K8`  — HKS[v, j] = Σ_i exp(-λ_i t_j) φ_i(v)² with the frozen analytic time grid.

**No checkpoints are read.** The script only needs the YAML configs in `configs/GPS/` (`pcqm4m-subset-GPS+LapPE.yaml` and `pcqm4m-subset-GPS+LHKS-K16-MLP3-fixed.yaml`) so `compute_posenc_stats` truncates to the right rank. No trained parameter enters either side of the comparison.

**Prerequisites**
1. `MyDrive/datasets/pcqm4m-v2/processed/` contains the OGB-processed PCQM4Mv2 `.pt` file (run `00_setup.ipynb` once).
2. The repo itself (cloned in step 4) — this is the only thing beyond the dataset.

**Drive account:** **`znidar.mark@gmail.com`** (authorize in the Drive popup).

**Time budget:** CPU-bound (no forward passes). 5000 graphs × 3 k_remove × 20 perturbations ≈ 30–60 min per method on a Colab CPU runtime. Use `N_GRAPHS = 500` first as a sanity check.

## 1. Configuration

Only two knobs: how many graphs to evaluate and which Drive account to use for outputs.

In [ ]:
# EDIT: smaller number (e.g. 500) for a quick sanity check; 5000 for final figure.
N_GRAPHS = 5000

GITHUB_REPO     = "https://github.com/mark-znidar/gdl__2.git"
DRIVE_WORKSPACE = "/content/drive/MyDrive"

DRIVE_DATASETS    = f"{DRIVE_WORKSPACE}/datasets"
DRIVE_EXP_RESULTS = f"{DRIVE_WORKSPACE}/pe_stability_results/exp1_pe_drift"

import os
os.makedirs(DRIVE_EXP_RESULTS, exist_ok=True)

print(f"N_GRAPHS          = {N_GRAPHS}")
print(f"Drive output dir  = {DRIVE_EXP_RESULTS}")

## 2. GPU check

This experiment does **no forward passes** — a CPU runtime is fine.

In [ ]:
!nvidia-smi | head -20 || echo '(no GPU detected; CPU is OK for this experiment)'

## 3. Mount Drive

Same procedure as the other runner notebooks; retries once with `force_remount=True` on first failure.

In [ ]:
from google.colab import drive
import os, shutil, time

MOUNTPOINT = "/content/drive"
MOUNT_TIMEOUT_MS = 180_000

def _clean_mountpoint():
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    if os.path.islink(MOUNTPOINT):
        os.remove(MOUNTPOINT)
    elif os.path.isdir(MOUNTPOINT) and os.listdir(MOUNTPOINT):
        shutil.rmtree(MOUNTPOINT)
    os.makedirs(MOUNTPOINT, exist_ok=True)

def _mount(force_remount=False):
    drive.mount(MOUNTPOINT, timeout_ms=MOUNT_TIMEOUT_MS,
                force_remount=force_remount)

_clean_mountpoint()
try:
    _mount()
except ValueError as e:
    if "mount failed" not in str(e).lower():
        raise
    print("=== Google Drive: mount failed ===\nRetrying once in 5s with force_remount=True ...\n", flush=True)
    time.sleep(5)
    _clean_mountpoint()
    _mount(force_remount=True)

## 4. Clone repo + install deps + symlink Drive folders

Dataset and the output folder (`MyDrive/pe_stability_results/exp1_pe_drift`) are symlinked into the repo so every write goes to Drive and survives VM restarts. No `results_pcqm4m_subset/` symlink is needed — this experiment does not read any checkpoints.

In [ ]:
import os, subprocess
%cd /content
if not os.path.isdir("gdl__2"):
    !git clone {GITHUB_REPO}
else:
    %cd gdl__2
    !git pull --ff-only || true
    %cd /content
%cd gdl__2
!bash run_colab/setup.sh

!rm -rf datasets
!ln -sfn {DRIVE_DATASETS} datasets
!mkdir -p ANALYSIS/perturbation
!rm -rf ANALYSIS/perturbation/exp1_pe_drift_results
!ln -sfn {DRIVE_EXP_RESULTS} ANALYSIS/perturbation/exp1_pe_drift_results

!ls -la datasets ANALYSIS/perturbation/exp1_pe_drift_results

## 5. Sanity checks — dataset + config files exist

Confirms the dataset is readable and the two YAMLs the script depends on are present in the repo.

In [ ]:
from pathlib import Path

proc = Path(DRIVE_DATASETS) / "pcqm4m-v2" / "processed"
assert proc.is_dir(), f"Missing {proc}. Run 00_setup.ipynb first."
big = [p for p in proc.glob("*.pt") if p.stat().st_size > 500_000_000]
assert big, "No large processed .pt found in MyDrive/datasets/pcqm4m-v2/processed/"
for p in big:
    print(f"OK dataset  {p.name}  {p.stat().st_size / 1e9:.1f} GB")

REQUIRED_CFGS = [
    "configs/GPS/pcqm4m-subset-GPS+LapPE.yaml",
    "configs/GPS/pcqm4m-subset-GPS+LHKS-K16-MLP3-fixed.yaml",
]
missing = [p for p in REQUIRED_CFGS if not Path(p).is_file()]
for p in REQUIRED_CFGS:
    tag = "OK config " if Path(p).is_file() else "[MISS]    "
    print(f"{tag} {p}")
if missing:
    raise FileNotFoundError(f"{len(missing)} config(s) missing: {missing}")

## 6. Run the experiment

Invokes the script once with `--all`. It iterates over the two methods, writes one JSON per method (`<method>.json`) into `ANALYSIS/perturbation/exp1_pe_drift_results/` (Drive-backed), and skips any method whose JSON already exists so interrupts are safe.

**Progress:** prints `[exp1-pe] 100/5000 ...` every 100 graphs, and prints a bin-validation table with `mean δ_min` and `mean #non-bridges` per bin A/B/C/D at startup.

In [ ]:
import subprocess, datetime, shlex, sys

SCRIPT = "ANALYSIS/perturbation/pe_stability_exp1_pe_drift.py"
cmd = ["python", "-u", SCRIPT,
       "--all",
       "--n-graphs", str(N_GRAPHS),
       "--no-plot"]
print(f">>> start: {datetime.datetime.now().isoformat(timespec='seconds')}")
print(f">>> cmd:   {' '.join(shlex.quote(c) for c in cmd)}", flush=True)

ret = subprocess.call(cmd)
print(f">>> done:  {datetime.datetime.now().isoformat(timespec='seconds')}  [{'OK' if ret == 0 else f'FAILED (exit {ret})'}]")
if ret != 0:
    raise RuntimeError(f"Exp1 PE-drift failed (exit {ret}). See output above.")

## 7. Aggregate + plot

`--plot-only` aggregates the two `*.json` files into `summary.json` (including `bin_stats` with per-bin `mean_nonbridges`) and writes `main.png`.

In [ ]:
!python ANALYSIS/perturbation/pe_stability_exp1_pe_drift.py --plot-only

In [ ]:
from IPython.display import Image, display
import json
from pathlib import Path

out = Path("ANALYSIS/perturbation/exp1_pe_drift_results")
png = out / "main.png"
if png.exists():
    display(Image(filename=str(png)))
else:
    print(f"(not found: {png})")

summary_path = out / "summary.json"
if summary_path.exists():
    s = json.load(open(summary_path))
    print("\n--- summary.bin_stats ---")
    print(json.dumps(s.get("bin_stats", {}), indent=2))
    print("\n--- summary.meta ---")
    print(json.dumps(s.get("meta", {}), indent=2))

## 8. Verify results on Drive

Everything under `ANALYSIS/perturbation/exp1_pe_drift_results/` lives on Drive via the symlink — safe across VM restarts.

In [ ]:
!ls -la {DRIVE_EXP_RESULTS}
!du -sh {DRIVE_EXP_RESULTS}